In [ ]:
import pandas as pd

# Membaca dataset hasil preprocessing
df = pd.read_csv('/content/dataset_timnas.csv')

# Menampilkan 5 baris pertama
display(df.head())

,text_raw,text,created_at,like_count
0,"@googlemoody padahal, banner tersebut menjelas...",padahal banner tersebut menjelaskan tentang ke...,2026-10-08T18:30:01.000Z,0
1,@googlemoody yang rame-rame saat ini setelah t...,yang rame rame saat ini setelah timnas indones...,2026-10-08T18:24:41.000Z,1
2,TC Piala Asia 2027 Dimulai Kapan? Ini Rencana ...,tc piala asia 2027 dimulai kapan ini rencana j...,2026-10-08T18:01:26.000Z,0
3,Calvin Verdonk Ditunggu Arsenal Usai Antar Tim...,calvin verdonk ditunggu arsenal usai antar tim...,2026-10-08T17:48:27.000Z,0
4,@IjoelSi @FaktaSepakbola Tapi bagi pemain Timn...,tapi bagi pemain timnas indonesia 10 000 itu d...,2026-10-08T17:34:38.000Z,0


In [ ]:
# 1. Memeriksa jumlah baris dan kolom
print("Jumlah baris dan kolom:", df.shape)

# 2. Memeriksa nama kolom
print("\nNama kolom:")
print(df.columns.tolist())

# 3. Memeriksa data kosong
print("\nJumlah data kosong:")
print(df.isnull().sum())

# 4. Memeriksa jumlah data duplikat
print("\nJumlah data duplikat:")
print(df.duplicated(subset=["text"]).sum())

Jumlah baris dan kolom: (284, 4)

Nama kolom:
['text_raw', 'text', 'created_at', 'like_count']

Jumlah data kosong:
text_raw      0
text          0
created_at    0
like_count    0
dtype: int64

Jumlah data duplikat:
0


In [ ]:
# LANGKAH 5: PROSES TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

# Daftar kata umum yang tidak banyak membantu
# membedakan topik postingan
stopwords_indonesia = [
    "yang", "dan", "di", "ke", "dari", "ini", "itu",
    "untuk", "dengan", "pada", "dalam", "karena",
    "jadi", "akan", "ada", "atau", "juga", "sudah",
    "belum", "masih", "saat", "saja", "lebih", "bisa",
    "aku", "saya", "kamu", "dia", "mereka", "kita",
    "kami", "nya", "lah", "kan", "nih", "sih", "aja",
    "bang", "dong", "deh", "yg", "dgn", "buat",
    "tentang", "terhadap", "sebagai", "ketika",
    "kalau", "kalo", "tp", "tapi", "namun",
    "the", "and", "is", "are", "of", "to", "in",
    "on", "for", "with", "at", "by", "from"
]

# Membuat objek TF-IDF
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words=stopwords_indonesia,
    max_features=3000,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)

# Mengubah teks menjadi matriks angka
matriks_tfidf = tfidf.fit_transform(df["text"])

# Menampilkan hasil
print("TF-IDF berhasil dibuat!")
print("Jumlah dokumen:", matriks_tfidf.shape[0])
print("Jumlah fitur kata/frasa:", matriks_tfidf.shape[1])

# Menampilkan beberapa kata yang berhasil dikenali
print("\nContoh fitur TF-IDF:")
print(tfidf.get_feature_names_out()[:30])

TF-IDF berhasil dibuat!
Jumlah dokumen: 284
Jumlah fitur kata/frasa: 3000

Contoh fitur TF-IDF:
['aa keren' 'aamiin' 'abis' 'abis cerai' 'abis timnas' 'abis udah' 'about'
 'about skin' 'abroad' 'abroad ga' 'abroad semua' 'acara' 'acara arak'
 'actorjaeminwinssda' 'actorjaeminwinssda outstanding' 'ad' 'adalah'
 'adaptif' 'adaptif hadapi' 'adu' 'afc' 'afc women' 'aff' 'agar' 'ajang'
 'akhir' 'akhirnya' 'akhirnya juara' 'akun' 'alaikum']


In [ ]:
# LANGKAH 6: CLUSTERING MENGGUNAKAN K-MEANS

from sklearn.cluster import KMeans

# Menentukan jumlah cluster
jumlah_cluster = 3

# Membuat model K-Means
model_kmeans = KMeans(
    n_clusters=jumlah_cluster,
    random_state=42,
    n_init=10
)

# Melakukan clustering
df["cluster"] = model_kmeans.fit_predict(matriks_tfidf)

# Menampilkan jumlah postingan setiap cluster
jumlah_per_cluster = df["cluster"].value_counts().sort_index()

print("HASIL CLUSTERING TIMNAS INDONESIA")
print("=" * 40)

for cluster_id, jumlah in jumlah_per_cluster.items():
    print(f"Cluster {cluster_id + 1}: {jumlah} postingan")

print("\nTotal postingan:", len(df))
print("Jumlah cluster:", df["cluster"].nunique())

HASIL CLUSTERING TIMNAS INDONESIA
Cluster 1: 57 postingan
Cluster 2: 168 postingan
Cluster 3: 59 postingan

Total postingan: 284
Jumlah cluster: 3


In [ ]:
# LANGKAH 7: MENCARI 5 KATA DOMINAN SETIAP CLUSTER

import pandas as pd

# Mengambil nama fitur dari TF-IDF
nama_fitur = tfidf.get_feature_names_out()

# Mengambil pusat masing-masing cluster
pusat_cluster = model_kmeans.cluster_centers_

# Menyimpan kata dominan setiap cluster
kata_dominan = {}

print("5 KATA DOMINAN SETIAP CLUSTER")
print("=" * 50)

for cluster_id in range(3):

    # Mengurutkan fitur berdasarkan bobot tertinggi
    indeks_teratas = pusat_cluster[cluster_id].argsort()[::-1][:5]

    # Mengambil lima kata dengan bobot tertinggi
    daftar_kata = [
        nama_fitur[indeks]
        for indeks in indeks_teratas
    ]

    kata_dominan[cluster_id] = daftar_kata

    print(f"\nCluster {cluster_id + 1}")
    print("Kata dominan:", ", ".join(daftar_kata))

5 KATA DOMINAN SETIAP CLUSTER

Cluster 1
Kata dominan: timnasday, persebaya, gibran, lulusan, smp

Cluster 2
Kata dominan: indonesia, timnas, timnas indonesia, sama, sepak

Cluster 3
Kata dominan: garuda, timnas garuda, timnas, selamat, juara


In [ ]:
# LANGKAH 8: MENAMPILKAN CONTOH POSTINGAN

for cluster_id in range(3):

    anggota_cluster = df[df["cluster"] == cluster_id]

    contoh_postingan = anggota_cluster.head(3)

    print("=" * 60)
    print(f"CLUSTER {cluster_id + 1}")
    print("=" * 60)

    for nomor, (_, baris) in enumerate(
        contoh_postingan.iterrows(), start=1
    ):
        print(f"\nContoh {nomor}:")
        print(baris["text_raw"])
        print("-" * 40)

CLUSTER 1

Contoh 1:
Bocil handuk putih viral

https://t.co/9suELXXS8P

https://t.co/wpbSLMYVhL

https://t.co/aXZzrjet0w

https://t.co/WyqTtFq8ee

Kamis Budi Peralta Persib Malam Jumat #EmpatMusimPertiwi Biasakan Klok Gabon #TimnasDay https://t.co/K7r1ZBHViT
----------------------------------------

Contoh 2:
AUBAMEANG PASTI BANGGA!!!💙🇬🇦
#Timnasday
----------------------------------------

Contoh 3:
Vibesnya jadi macam pildun 🇮🇩🦅🏆

 #KitaGaruda #TimnasIndonesia #TimnasDay #FIFAASEANCup2026 https://t.co/jL7t5K4NPJ
----------------------------------------
CLUSTER 2

Contoh 1:
@googlemoody padahal, banner tersebut menjelaskan tentang keresahan dan kekecewaan sekelompok bobotoh pada negara / timnas indonesia.

hanya persib, yang bikin mereka hidup lebih ceria, bersemangat, hingga meneteskan air mata.

persib is more valuable than anything.
----------------------------------------

Contoh 2:
@googlemoody yang rame-rame saat ini setelah timnas indonesia berhasil menjuarai Asean Fifa Cup 2026

In [ ]:
# LANGKAH 9: MEMBERIKAN NAMA TEMA CLUSTER

# Nama tema berdasarkan pemeriksaan awal contoh postingan.
nama_tema = {
    0: "Percakapan Beragam Seputar Sepak Bola",
    1: "Isu dan Perbincangan Seputar Timnas Indonesia",
    2: "Prestasi dan Apresiasi Timnas Indonesia"
}

# Menambahkan nama tema pada setiap postingan.
df["tema_cluster"] = df["cluster"].map(nama_tema)

# Menambahkan lima kata dominan setiap cluster.
df["kata_dominan"] = df["cluster"].map(
    lambda x: ", ".join(kata_dominan[x])
)

# Menampilkan jumlah postingan setiap tema.
print("JUMLAH POSTINGAN PER TEMA")
print("=" * 55)

print(
    df.groupby("tema_cluster")
    .size()
    .sort_values(ascending=False)
)

# Menampilkan contoh hasil.
print("\nCONTOH HASIL CLUSTERING")

display(
    df[
        ["text_raw", "cluster", "kata_dominan", "tema_cluster"]
    ].head(10)
)

JUMLAH POSTINGAN PER TEMA
tema_cluster
Isu dan Perbincangan Seputar Timnas Indonesia    168
Prestasi dan Apresiasi Timnas Indonesia           59
Percakapan Beragam Seputar Sepak Bola             57
dtype: int64

CONTOH HASIL CLUSTERING


,text_raw,cluster,kata_dominan,tema_cluster
0,"@googlemoody padahal, banner tersebut menjelas...",1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
1,@googlemoody yang rame-rame saat ini setelah t...,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
2,TC Piala Asia 2027 Dimulai Kapan? Ini Rencana ...,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
3,Calvin Verdonk Ditunggu Arsenal Usai Antar Tim...,2,"garuda, timnas garuda, timnas, selamat, juara",Prestasi dan Apresiasi Timnas Indonesia
4,@IjoelSi @FaktaSepakbola Tapi bagi pemain Timn...,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
5,@atmawiratna_ Terimakasih para pelatih yg tela...,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
6,@atmawiratna_ @grok berapa lama Alfred Riedl m...,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
7,@Rizfad37 @bolabol77 Ada bukti? Pikiran loe aj...,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
8,@whiskyouthh @agamzett Soalnya dia sendiri yan...,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
9,https://t.co/R0DPcaUAaS\n\nyang suka hujat tim...,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia


In [ ]:
# LANGKAH 10: MENYIMPAN HASIL CLUSTERING

# Menentukan kolom yang akan disimpan
kolom_output = [
    "text_raw",
    "text",
    "created_at",
    "like_count",
    "cluster",
    "kata_dominan",
    "tema_cluster"
]

# Membuat dataset hasil clustering
hasil_cluster = df[kolom_output].copy()

# Menyimpan ke file CSV
hasil_cluster.to_csv(
    "/content/hasil_cluster.csv",
    index=False,
    encoding="utf-8-sig"
)

print("HASIL CLUSTERING BERHASIL DISIMPAN!")
print("Nama file: hasil_cluster.csv")
print("Jumlah data:", len(hasil_cluster))
print("\nJumlah postingan per cluster:")

print(
    hasil_cluster["tema_cluster"]
    .value_counts()
)

# Menampilkan beberapa baris hasil
display(hasil_cluster.head())

HASIL CLUSTERING BERHASIL DISIMPAN!
Nama file: hasil_cluster.csv
Jumlah data: 284

Jumlah postingan per cluster:
tema_cluster
Isu dan Perbincangan Seputar Timnas Indonesia    168
Prestasi dan Apresiasi Timnas Indonesia           59
Percakapan Beragam Seputar Sepak Bola             57
Name: count, dtype: int64


,text_raw,text,created_at,like_count,cluster,kata_dominan,tema_cluster
0,"@googlemoody padahal, banner tersebut menjelas...",padahal banner tersebut menjelaskan tentang ke...,2026-10-08T18:30:01.000Z,0,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
1,@googlemoody yang rame-rame saat ini setelah t...,yang rame rame saat ini setelah timnas indones...,2026-10-08T18:24:41.000Z,1,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
2,TC Piala Asia 2027 Dimulai Kapan? Ini Rencana ...,tc piala asia 2027 dimulai kapan ini rencana j...,2026-10-08T18:01:26.000Z,0,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia
3,Calvin Verdonk Ditunggu Arsenal Usai Antar Tim...,calvin verdonk ditunggu arsenal usai antar tim...,2026-10-08T17:48:27.000Z,0,2,"garuda, timnas garuda, timnas, selamat, juara",Prestasi dan Apresiasi Timnas Indonesia
4,@IjoelSi @FaktaSepakbola Tapi bagi pemain Timn...,tapi bagi pemain timnas indonesia 10 000 itu d...,2026-10-08T17:34:38.000Z,0,1,"indonesia, timnas, timnas indonesia, sama, sepak",Isu dan Perbincangan Seputar Timnas Indonesia


In [ ]:
from google.colab import files

files.download("/content/hasil_cluster.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>